---
title: The Coalescent
---

In [ ]:
# #| echo: false
# #| column: margin
# from phasic.utils import download_link
# download_link(__vsc_ipynb_file__) 

In [ ]:
from phasic import Graph
from phasic.utils import tree_illustration
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from vscodenb import set_vscode_theme
set_vscode_theme()


<!-- 

**Introduction:** The coalescent process as an example of a Markov jump process.

**State-space construction:** Multiple approaches to building state spaces: manual construction, callback functions, and matrix interfaces.

**Visualizing state space:** Plot state space graphs for visual inspection.

**Moments and distributions:** Computing moments, expectations, variances, PDFs, CDFs, and reward transformations.

**Discrete phase-type distributions:** Discretization of continuous models, discrete PMFs and CDFs, and multivariate marginal distributions.

**State vector indexing:** The state indexing utility for mapping between flat indices and structured state properties.

**Parameterized models:** Parameterizing edge weights to evaluate models across different parameter values without graph reconstruction.

### Computation

**MPFR:** Automated use of multiple-precision floating-point computation.

**Hierarchical symbolic trace:** Trace-based computation for efficient repeated evaluation of parameterized models.

**Multi-level caching:** Runtime and persistent caching of graphs and models.

**Model sharing:** Sharing elimination traces via IPFS for community reuse.

**Distributed computing:** Scaling inference across multiple devices on a SLURM cluster.


### Inference

**Method of Moments:** Data-informed priors via moment matching for use with SVGD inference.

**Inference basics:** Bayesian parameter inference using SVGD with priors, learning rate schedules, regularization, and adaptive optimizers.

**Multi-feature inference:** SVGD on reward-transformed observations with multiple features using dense and sparse observation formats.

**Multi-parameter inference:** Fitting models with multiple parameters, demonstrated on a two-population island model.

**Joint probability inference:** SVGD inference on discrete joint probability observations including ARG models.

### Modelling

**Time inhomogeneity:** Step-wise construction of time-inhomogeneous processes using epoch-wise approaches.

**Laplace transform:** Laplace transform computation and its use for computing expectations and higher moments.

**Joint probability:** Joint probability distributions of multiple random variables from a single phase-type model.

**Finite Markov chains:** Distribution of steps spent in states and runs of particular states.
 -->

The sequence of tutorials show-cases the complete Python API of the Phasic library, a high-performance computational framework for modelling inference on waiting times in Markov jump processes. In addition to this introduction, the tutorial covers the following topics - in each as a jupyter notebook you can download and play with yourself.

Throughout the tutorials we use the Coalescent model as an example. The Coalescent is a continuous-time Markov chain that models the ancestry of a sample of individuals from a population. Lineages coalesce into fewer lineages with more descendants until only a single common ancestral lineage remains. Branches in the resulting tree represent lineages. We call those with a single sampled descendant singleton lineages because mutations occurring on those branches produce a singleton mutation in our sample. Similarly with doubletons tripleton and so on. 

In [ ]:
#| echo: false 
#| label: fig-coalescent-intro
#| fig-cap: "Mutations on branches with a single descendant produce singleton variants in the sample."

#tree_illustration('TTTTGGGGA') ;

In [ ]:
#| echo: false 
#| label: fig-coalescent-intro-branches
#| fig-cap: "Mutations on branches with a single descendant produce singleton variants in the sample."

from phasic.utils import tree_illustration
fig, axes = plt.subplots(1, 3, figsize=(8, 4))
tree_illustration('TTTTT', ton=1, ax=axes[0], seed=21)
tree_illustration('TTTTT', ton=2, ax=axes[1], seed=21)
tree_illustration('TTTTT', ton=3, ax=axes[2], seed=21) ;

We can encode the states of this Markov chain as vectors enumerating the number of live 1'tons, 2'tons, 3'tons, and 4'tons branches (@fig-coalescent-intro-branches). With four samples, the Coalescent jumps starts with a state of four singletons and terminates/absorbs at the state with a single 4'ton (@fig-coalescent-graph).

In [ ]:
#| echo: false 
#| label: fig-coalescent-graph 
#| fig-cap: "The coalescent process with four samples. The S (starting) state just signifies that the process begins in state (4,0,0,0). Edges are annotated with the coalescence rates that, in this model represents the transition rates of the model"

def coalescent(state):
    transitions = []
    for i in range(state.size):
        for j in range(i, state.size):            
            same = int(i == j)
            if same and state[i] < 2:
                continue
            if not same and (state[i] < 1 or state[j] < 1):
                continue 
            new = state.copy()
            new[i] -= 1
            new[j] -= 1
            new[i+j+1] += 1
            transitions.append((new, state[i]*(state[j]-same)/(1+same)))
    return transitions

nr_samples = 4
ipv = [([nr_samples]+[0]*(nr_samples-1), 1)]
graph = Graph(coalescent, ipv=ipv)

graph.plot()